In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

In [2]:
dataset_dir = Path("../dataset")
csv_files = sorted(dataset_dir.glob("*.csv"))


def detect_time_col(df: pd.DataFrame):
    for c in ["time", "date", "Date", "datetime", "Datetime"]:
        if c in df.columns:
            return c
    return None


def detect_return_col(df: pd.DataFrame):
    priority = ["return_1d", "return_1_day", "return_1d_vn30", "return_1d_vnindex"]
    for c in priority:
        if c in df.columns:
            return c
    for c in df.columns:
        lc = c.lower()
        if "return" in lc and ("1d" in lc or "1_day" in lc or "1day" in lc):
            return c
    return None


returns_by_dataset = {}
returns_time_by_dataset = {}

for file_path in csv_files:
    df = pd.read_csv(file_path)
    time_col = detect_time_col(df)
    return_col = detect_return_col(df)

    if file_path.name == "VN30_dataset_from_2019.csv":
        time_series = pd.to_datetime(df[time_col], errors="coerce")
        ret = pd.to_numeric(df["return_1d"], errors="coerce")
        mc = pd.to_numeric(df["Market Capital (Bn VND)"], errors="coerce")
        tmp = pd.DataFrame({"time": time_series, "ret": ret, "mc": mc}).dropna(subset=["time", "ret", "mc"])
        tmp = tmp[(tmp["time"].dt.year >= 2020) & (tmp["time"].dt.year <= 2025)]

        def weighted_mean(g: pd.DataFrame):
            w = g["mc"].to_numpy(dtype=float)
            x = g["ret"].to_numpy(dtype=float)
            valid = np.isfinite(w) & np.isfinite(x) & (w > 0)
            if valid.sum() == 0:
                return np.nan
            wv = w[valid]
            xv = x[valid]
            ws = wv.sum()
            if ws <= 0:
                return np.nan
            return float(np.sum(xv * (wv / ws)))

        stock_return = tmp.groupby("time", as_index=False).apply(weighted_mean, include_groups=False)
        stock_return = stock_return.rename(columns={None: "stock_return", 0: "stock_return"})
        stock_return = stock_return.dropna(subset=["stock_return"])
        s = stock_return["stock_return"].astype(float) * 100.0
        t = stock_return["time"]
        key = "VN30_stock_return"
    else:
        if return_col is None:
            continue
        if time_col is not None:
            t = pd.to_datetime(df[time_col], errors="coerce")
            mask = (t.dt.year >= 2020) & (t.dt.year <= 2025)
            s = pd.to_numeric(df.loc[mask, return_col], errors="coerce")
            t = t.loc[mask]
        else:
            s = pd.to_numeric(df[return_col], errors="coerce")
            t = pd.Series([pd.NaT] * len(s))
        s = s.replace([np.inf, -np.inf], np.nan).dropna().astype(float) * 100.0
        key = file_path.stem

    if len(s) < 20:
        continue
    returns_by_dataset[key] = s.reset_index(drop=True)
    returns_time_by_dataset[key] = t.reset_index(drop=True)

for dataset_name, series in returns_by_dataset.items():
    x = series.to_numpy(dtype=float)
    fig = go.Figure()
    fig.add_trace(
        go.Histogram(
            x=x,
            nbinsx=80,
            histnorm="probability density",
            name=dataset_name,
            opacity=0.85,
        )
    )
    fig.update_layout(
        title=f"Histogram return 1D (%) - {dataset_name}",
        xaxis_title="return (%)",
        yaxis_title="density",
        width=700,
        height=360,
        xaxis=dict(range=[-10, 10]),
        bargap=0.05,
    )
    fig.show()

In [3]:
student_t_rows = []

for dataset_name, series in returns_by_dataset.items():
    x = series.to_numpy(dtype=float)
    nu, loc, scale = stats.t.fit(x)
    ks_stat, p_value = stats.kstest(x, "t", args=(nu, loc, scale))
    student_t_rows.append(
        {
            "dataset": dataset_name,
            "n_obs": len(x),
            "nu": nu,
            "loc": loc,
            "scale": scale,
            "ks_stat": ks_stat,
            "p_value": p_value,
            "reject_H0_5pct": bool(p_value < 0.05),
        }
    )

student_t_results = pd.DataFrame(student_t_rows).sort_values("p_value", ascending=False)
student_t_results

,dataset,n_obs,nu,loc,scale,ks_stat,p_value,reject_H0_5pct
4,Nikkei_225,1465,4.157325,0.075569,0.986405,0.011515,0.988904,False
5,SMI,1510,3.381797,0.053074,0.627187,0.015056,0.878174,False
6,snp500,1508,2.847071,0.101773,0.762399,0.015576,0.852029,False
0,DAX_40,1528,2.911452,0.090378,0.765298,0.015696,0.839925,False
1,EuroNext_100,1538,2.961028,0.086785,0.701619,0.016930,0.763394,False
3,KOSPI_index,1471,4.416500,0.083457,0.946155,0.017801,0.732739,False
7,VN30_stock_return,1499,2.520291,0.159579,0.771783,0.018489,0.677608,False
8,VN30_INDEX,1499,2.441149,0.153697,0.782911,0.022427,0.431660,False
2,IBEX_35,1542,3.556553,0.096361,0.823466,0.022734,0.396974,False
9,VN_INDEX,1499,2.393271,0.167796,0.717454,0.025509,0.278753,False


In [4]:
nu_table = student_t_results[["dataset", "nu"]].sort_values("nu", ascending=False).reset_index(drop=True)
nu_table

,dataset,nu
0,KOSPI_index,4.416500
1,Nikkei_225,4.157325
2,IBEX_35,3.556553
3,SMI,3.381797
4,EuroNext_100,2.961028
5,DAX_40,2.911452
6,snp500,2.847071
7,VN30_stock_return,2.520291
8,VN30_INDEX,2.441149
9,VN_INDEX,2.393271


In [5]:
risk_rows = []

for dataset_name, series in returns_by_dataset.items():
    x = series.dropna().astype(float)
    if len(x) < 20:
        continue

    mu = float(x.mean())
    sigma = float(x.std(ddof=1))
    skewness = float(stats.skew(x, bias=False))
    kurt = float(stats.kurtosis(x, fisher=True, bias=False))

    var_95 = float(np.percentile(x, 5))
    tail = x[x <= var_95]
    es_95 = float(tail.mean()) if len(tail) > 0 else np.nan

    wealth = (1.0 + x / 100.0).cumprod()
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1.0
    max_drawdown_pct = float(drawdown.min() * 100.0)

    risk_rows.append(
        {
            "dataset": dataset_name,
            "mu_mean_return_pct": mu,
            "sigma_volatility_pct": sigma,
            "skewness": skewness,
            "kurtosis": kurt,
            "VaR_95_pct": var_95,
            "ES_CVaR_95_pct": es_95,
            "max_drawdown_pct": max_drawdown_pct,
        }
    )

risk_table_2020_2025 = pd.DataFrame(risk_rows).sort_values("sigma_volatility_pct", ascending=False).reset_index(drop=True)
risk_table_2020_2025

,dataset,mu_mean_return_pct,sigma_volatility_pct,skewness,kurtosis,VaR_95_pct,ES_CVaR_95_pct,max_drawdown_pct
0,Nikkei_225,0.051546,1.396142,-0.381468,10.372172,-2.071117,-3.149270,-31.861626
1,VN30_INDEX,0.055854,1.381058,-0.885225,4.410993,-2.299412,-3.820149,-44.101228
2,VN30_stock_return,0.062050,1.352698,-0.938357,5.138410,-2.244120,-3.749914,-35.790050
3,snp500,0.049792,1.321852,-0.646549,14.686786,-1.855694,-3.217151,-36.102640
4,VN_INDEX,0.041289,1.294794,-1.051791,5.026584,-2.228904,-3.670317,-41.833897
5,KOSPI_index,0.044259,1.292230,-0.377947,6.240601,-1.949015,-3.056479,-36.535497
6,DAX_40,0.040207,1.278866,-0.710076,13.492614,-1.830314,-3.155861,-39.984623
7,IBEX_35,0.038567,1.269732,-1.419691,18.501578,-1.755772,-3.057431,-40.916542
8,EuroNext_100,0.026518,1.170819,-1.267708,14.706120,-1.696015,-2.993292,-39.131405
9,SMI,0.014759,0.977621,-1.095565,12.977040,-1.343354,-2.417961,-28.586877
